# Fair Code — Audit 03: German Credit Lending Bias

> *Young applicants (<30) were denied good-credit ratings at significantly higher rates than older applicants with identical financial profiles. The algorithm wasn't told to discriminate by age. It learned to.*

**Dataset:** German Credit Risk dataset — `credit_customers.csv`  
**Protected attribute:** Age (young = <30)  
**Proxy variable:** `employment` (career tenure is a direct function of age)  
**Fairness metric:** Demographic Parity  
**Model:** Random Forest Classifier  

---

Pipeline:
1. Load and explore the dataset
2. Identify the proxy variable (`employment`)
3. Train the biased model (age + proxy included)
4. Measure the fairness gap
5. Train the fair model (both removed)
6. Compare results

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency

# Consistent styling across all Fair Code notebooks
plt.rcParams.update({
    'figure.facecolor': '#0d0f14',
    'axes.facecolor': '#131620',
    'axes.edgecolor': '#1e2130',
    'axes.labelcolor': '#b0aec0',
    'xtick.color': '#b0aec0',
    'ytick.color': '#b0aec0',
    'text.color': '#d4cfc0',
    'grid.color': '#1e2130',
    'grid.linestyle': '--',
    'font.family': 'monospace',
    'figure.dpi': 120
})

ACCENT = '#c9a84c'   # Fair Code gold
DANGER = '#9b2335'   # red — bias
SAFE   = '#4a7c6f'   # teal — mitigated
MUTED  = '#b0aec0'

print('Libraries loaded.')

## 2. Load and Explore the Dataset

In [ ]:
df_raw = pd.read_csv('German Credit Lending/credit_customers.csv')
print(f'Dataset: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')
print(f'\nColumns: {list(df_raw.columns)}')
df_raw.head(3)

In [ ]:
df = df_raw.copy()

# Target: 1 = good credit, 0 = bad credit
df['target'] = (df['class'] == 'good').astype(int)

# Protected attribute: age — young applicants flagged as bad credit
# at significantly higher rates despite equivalent financial profiles
df['is_young'] = (df['age'] < 30).astype(int)

print('Age group breakdown:')
print(df['is_young'].value_counts().rename({0: 'Older (30+)', 1: 'Young (<30)'}).to_string())

print(f'\nOverall good-credit rate: {df["target"].mean():.1%}')
print('\nGood-credit rate by age group:')
rates = df.groupby('is_young')['target'].mean() * 100
print(f'  Older (30+): {rates[0]:.1f}%')
print(f'  Young (<30): {rates[1]:.1f}%')
print(f'\nRaw gap in dataset: {rates[0] - rates[1]:.2f} percentage points')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# Good-credit rate by age group
labels = ['Older (30+)', 'Young (<30)']
vals = [rates[0], rates[1]]
colors = [MUTED, DANGER]
bars = axes[0].bar(labels, vals, color=colors, width=0.4)
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.5,
                 f'{val:.1f}%', ha='center', color=ACCENT, fontsize=11)
axes[0].set_title('Good-credit rate by age group (raw data)', color=MUTED, fontsize=10)
axes[0].set_ylabel('Good-credit rate (%)')
axes[0].set_ylim(0, 90)
axes[0].grid(axis='y', alpha=0.3)

# Credit amount distribution by age group
for label, mask, color in [
    ('Older (30+)', df['is_young'] == 0, MUTED),
    ('Young (<30)', df['is_young'] == 1, ACCENT)
]:
    axes[1].hist(df[mask]['credit_amount'], bins=25, alpha=0.6, color=color, label=label)
axes[1].set_title('Credit amount by age group', color=MUTED, fontsize=10)
axes[1].set_xlabel('Credit amount (DM)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print('Note: credit amount distributions overlap substantially — the age gap is not explained by loan size.')

## 3. Identify the Proxy Variable

A proxy variable correlates with a protected attribute even though it doesn't name it directly.
Removing `age` alone isn't enough — the model will reconstruct the age signal through correlated features.

**Candidate:** `employment` — career tenure is a direct function of age.  
Young applicants (<30) have <1yr employment at **27.2%** vs only **11.3%** for older applicants.  
Older applicants have ≥7yr employment at **35.9%** vs just **7.3%** for young applicants.

In [ ]:
def check_proxy(df, feature, protected_col):
    """Chi-squared test of independence. p < 0.05 = likely proxy."""
    contingency = pd.crosstab(df[feature], df[protected_col])
    chi2, p, dof, _ = chi2_contingency(contingency)
    return {'feature': feature, 'protected_attr': protected_col,
            'p_value': round(p, 6), 'is_proxy': p < 0.05}

result = check_proxy(df, 'employment', 'is_young')
print(f"Feature:        {result['feature']}")
print(f"Protected attr: age (is_young)")
print(f"p-value:        {result['p_value']}")
print(f"Is proxy:       {result['is_proxy']}")

In [ ]:
ct = pd.crosstab(df['employment'], df['is_young'].map({0: 'Older (30+)', 1: 'Young (<30)'}),
                 normalize='columns').round(3)
print('Employment tenure distribution by age group (column proportions):')
print(ct.to_string())
print('\nConclusion: tenure patterns differ dramatically by age.')
print('Including employment lets the model infer age even without the age column.')

In [ ]:
# Visualise the employment-age correlation
ct_plot = pd.crosstab(df['employment'], df['is_young'].map({0: 'Older (30+)', 1: 'Young (<30)'}),
                      normalize='columns') * 100

x = np.arange(len(ct_plot.index))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width/2, ct_plot['Older (30+)'], width, color=MUTED, label='Older (30+)', alpha=0.85)
ax.bar(x + width/2, ct_plot['Young (<30)'], width, color=DANGER, label='Young (<30)', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(ct_plot.index, rotation=20, ha='right')
ax.set_ylabel('Share of group (%)')
ax.set_title('Employment tenure by age group — proxy signal', color=ACCENT, pad=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Train the Biased Model

Features include `age` directly **and** `employment` as a proxy.

In [ ]:
# Encode categoricals
cat_cols = [
    'checking_status', 'credit_history', 'purpose', 'savings_status',
    'employment', 'personal_status', 'other_parties', 'property_magnitude',
    'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker'
]
df_enc = df.copy()
le = LabelEncoder()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col])

print('Categorical columns encoded.')

In [ ]:
biased_features = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings_status',
    'employment',            # proxy variable for age
    'installment_commitment', 'personal_status', 'other_parties', 'residence_since',
    'property_magnitude',
    'age',                   # protected attribute
    'other_payment_plans', 'housing', 'existing_credits', 'job',
    'num_dependents', 'own_telephone', 'foreign_worker',
]

X_biased = df_enc[biased_features]
y = df_enc['target']

X_train, X_test, y_train, y_test = train_test_split(
    X_biased, y, test_size=0.2, random_state=42
)

biased_model = RandomForestClassifier(n_estimators=100, random_state=42)
biased_model.fit(X_train, y_train)
biased_preds = biased_model.predict(X_test)
biased_accuracy = accuracy_score(y_test, biased_preds)

results_b = X_test.copy()
results_b['pred']     = biased_preds
results_b['is_young'] = df_enc.loc[X_test.index, 'is_young'].values

rates_b = results_b.groupby('is_young')['pred'].mean() * 100
gap_b   = rates_b[0] - rates_b[1]

print('--- BIASED MODEL RESULTS ---')
print()
print(f"Older Applicants (30+) Good Credit Rate: {rates_b[0]:.2f}%")
print(f"Young Applicants (<30) Good Credit Rate: {rates_b[1]:.2f}%")
print()
print(f"Fairness Gap: {gap_b:.2f}%")
print(f"Model Accuracy: {biased_accuracy:.2%}")

## 5. Train the Fair Model

`age` and `employment` are both removed. Only objective financial signals remain.

In [ ]:
fair_features = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings_status',
    # employment removed ✓  (proxy: tenure correlates directly with age)
    'installment_commitment', 'personal_status', 'other_parties', 'residence_since',
    'property_magnitude',
    # age removed ✓  (protected attribute)
    'other_payment_plans', 'housing', 'existing_credits', 'job',
    'num_dependents', 'own_telephone', 'foreign_worker',
]

X_fair = df_enc[fair_features]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fair, y, test_size=0.2, random_state=42
)

fair_model = RandomForestClassifier(n_estimators=100, random_state=42)
fair_model.fit(X_train_f, y_train_f)
fair_preds = fair_model.predict(X_test_f)
fair_accuracy = accuracy_score(y_test_f, fair_preds)

results_f = X_test_f.copy()
results_f['pred']     = fair_preds
results_f['is_young'] = df_enc.loc[X_test_f.index, 'is_young'].values

rates_f = results_f.groupby('is_young')['pred'].mean() * 100
gap_f   = rates_f[0] - rates_f[1]

print('--- MITIGATED (UNBIASED) RESULTS ---')
print()
print(f"Older Applicants (30+) Good Credit Rate: {rates_f[0]:.2f}%")
print(f"Young Applicants (<30) Good Credit Rate: {rates_f[1]:.2f}%")
print()
print(f"New Fairness Gap: {gap_f:.2f}%")
print(f"Model Accuracy: {fair_accuracy:.2%}")

## 6. Compare Results

In [ ]:
reduction = (gap_b - gap_f) / gap_b * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('German Credit — Biased vs Mitigated Model', color=ACCENT, fontsize=13, y=1.02)

groups = ['Older (30+)', 'Young (<30)']

for ax, rates, colors, title in [
    (axes[0], rates_b, [MUTED, DANGER], f'Biased model\nGap: {gap_b:.2f}%'),
    (axes[1], rates_f, [SAFE,  SAFE  ], f'Mitigated model\nGap: {gap_f:.2f}%'),
]:
    vals = [rates[0], rates[1]]
    bars = ax.bar(groups, vals, color=colors, width=0.45)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.5,
                f'{val:.1f}%', ha='center', fontsize=11, color=ACCENT)
    ax.set_ylim(0, 100)
    ax.set_ylabel('Good-credit rate (%)')
    ax.set_title(title, color=MUTED, fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nSummary')
print(f'-------')
print(f'Fairness gap before: {gap_b:.2f}%')
print(f'Fairness gap after:  {gap_f:.2f}%')
print(f'Reduction:           {reduction:.1f}%')

## Key Insight

Removing `age` alone is not enough. `employment` tenure is a near-perfect proxy for age — a 25-year-old cannot have 7+ years of work history the way a 45-year-old can. The model learns this relationship from training data and reconstructs an age penalty through tenure even when the `age` column is absent.

**The fix:** Remove both the protected attribute **and** its proxy variables. Retain only objective financial signals — checking account status, credit history, loan purpose, savings — that reflect actual creditworthiness rather than who the applicant is.

---

*Part of the [Fair Code project](https://github.com/yakew7/Fair-Code) by [@thefaircodeproject](https://instagram.com/thefaircodeproject)*